<a href="https://colab.research.google.com/github/Hemant10HM/ANNDL-LAB_24mcs004/blob/main/ANN_LAB_8_ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Implementing ViT on pistachio dataset


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import timm
import random


In [ ]:
#configuration
DATA_DIR = "./Pistachio_Image_Dataset"
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
WEIGHT_DECAY = 1e-4
IMAGE_SIZE = 224
TRAIN_SPLIT = 0.8
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ==== SET RANDOM SEED ====
torch.manual_seed(SEED)
random.seed(SEED)

# ==== TRANSFORMS ====
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


In [ ]:
# ==== LOAD AND SPLIT DATASET ====
# Load entire dataset with basic transform first
full_dataset = datasets.ImageFolder(DATA_DIR)

# Get class names
class_names = full_dataset.classes

# Split indices
total_size = len(full_dataset)
train_size = int(TRAIN_SPLIT * total_size)
test_size = total_size - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# Apply transforms manually (ImageFolder with split loses transform logic)
train_dataset = [(train_transforms(img), label) for img, label in train_dataset]
test_dataset = [(test_transforms(img), label) for img, label in test_dataset]

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# ==== MODEL ====
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=len(class_names))
model.to(DEVICE)

# ==== LOSS & OPTIMIZER ====
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ==== TRAIN FUNCTION ====
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    return running_loss / len(loader), correct / len(loader.dataset)


c:\Users\heman\Desktop\Hemant Study\LNMIIT\Python Projects Environments\plant\plant\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\heman\.cache\huggingface\hub\models--timm--vit_base_patch16_224.augreg2_in21k_ft_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [ ]:
# ==== EVALUATE FUNCTION ====
def evaluate(model, loader, loss_fn, device):
    model.eval()
    running_loss = 0.0
    correct = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)

            running_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

    return running_loss / len(loader), correct / len(loader.dataset)

# ==== MAIN TRAINING LOOP ====
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, DEVICE)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")

# ==== SAVE MODEL ====
torch.save(model.state_dict(), "vit_pistachio.pth")
print("✅ Model saved as vit_pistachio.pth")


Epoch 1/10


Evaluating: 100%|██████████| 14/14 [00:10<00:00,  1.38it/s]


Train Loss: 0.8165 | Train Acc: 0.5192
Test  Loss: 0.7008 | Test  Acc: 0.4070

Epoch 2/10


Evaluating: 100%|██████████| 14/14 [00:10<00:00,  1.38it/s]


Train Loss: 0.6647 | Train Acc: 0.5949
Test  Loss: 0.6390 | Test  Acc: 0.6116

Epoch 3/10


...

Train Loss: 0.2327 | Train Acc: 0.9034
Test  Loss: 0.3985 | Test  Acc: 0.8302

Epoch 9/10


Evaluating: 100%|██████████| 14/14 [00:10<00:00,  1.37it/s]


Train Loss: 0.1833 | Train Acc: 0.9325
Test  Loss: 0.2285 | Test  Acc: 0.8907

Epoch 10/10


Evaluating: 100%|██████████| 14/14 [00:10<00:00,  1.37it/s]


Train Loss: 0.1089 | Train Acc: 0.9610
Test  Loss: 0.2865 | Test  Acc: 0.8860
✅ Model saved as vit_pistachio.pth
